In [1]:
# 1: Mount Google Drive, load the NEW JSON file, and sanity-check size

from google.colab import drive
from pathlib import Path
import json

drive.mount("/content/drive")

INPUT_PATH = Path("/content/drive/MyDrive/final_project/2wikimultihopqa_dev_2020wiki_1000_fullwiki_clean_postproc_v2.json")
assert INPUT_PATH.exists(), f"File not found: {INPUT_PATH}"

with INPUT_PATH.open("r", encoding="utf-8") as f:
    data = json.load(f)

if not isinstance(data, list):
    print(f"Unexpected type: {type(data)} (expected list)")
else:
    print(f"Number of examples in JSON: {len(data)}")
    if len(data) == 1000:
        print("OK: JSON contains 1000 examples.")
    else:
        print("Warning: JSON does NOT contain 1000 examples.")

Mounted at /content/drive
Number of examples in JSON: 1000
OK: JSON contains 1000 examples.


In [2]:
# 2: Show one example to inspect the structure / keys

from pprint import pprint

print("example index:", 0)
print("Top-level keys:", list(data[0].keys()))

pprint(data[0], width=140, sort_dicts=False)


example index: 0
Top-level keys: ['_id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context', 'titles', 'docs', 'docs2', 'evidences']
{'_id': 'a7b9672009c311ebbdb0ac1f6bf848b6',
 'question': 'Are North Marion High School (Oregon) and Seoul High School both located in the same country?',
 'answer': 'no',
 'type': 'comparison',
 'level': None,
 'supporting_facts': [['North Marion High School (Oregon)', 0], ['Seoul High School', 0]],
 'context': [['Calloway County High School',
              ['Calloway County',
               'High School is a public high school located in Murray, Kentucky.',
               'The school was formed from the consolidation of six high schools from across the county: Hazel High School, Lynn Grove '
               'High School, Kirksey High School, Almo High School, New Concord High School, and Faxon High School.']],
             ['Marion High School (Kansas)',
              ['Marion High School is a public high school in Marion, Kansas, USA.',

In [3]:
# 3: Show one random example and compare context vs docs per title

import random
from pprint import pprint

idx = random.randrange(len(data))
ex = data[idx]

print(f"Random example index: {idx}")
print("=" * 80)
print(f"_id:      {ex.get('_id')}")
print(f"type:     {ex.get('type')}")
print(f"level:    {ex.get('level')}")
print(f"question: {ex.get('question')}")
print(f"answer:   {ex.get('answer')}")
print("=" * 80)

print("Supporting facts:")
pprint(ex.get("supporting_facts"), width=140, sort_dicts=False)
print("=" * 80)

# Build title -> context paragraph
context_map = {}
for item in ex.get("context", []):
    if isinstance(item, (list, tuple)) and len(item) == 2:
        ctx_title, sent_list = item
        if isinstance(ctx_title, str) and isinstance(sent_list, list):
            paragraph = " ".join(s.strip() for s in sent_list if isinstance(s, str) and s.strip())
            context_map[ctx_title] = paragraph

titles = ex.get("titles", [])
docs = ex.get("docs", [])

if not isinstance(titles, list): titles = []
if not isinstance(docs, list): docs = []

# Align docs length to titles (avoid zip truncation hiding problems)
if len(docs) < len(titles):
    docs = docs + [""] * (len(titles) - len(docs))
elif len(docs) > len(titles):
    docs = docs[:len(titles)]

max_chars = 2000

print("Titles, context paragraphs and docs (aligned by title):\n")

for i, (t, d) in enumerate(zip(titles, docs)):
    print(f"[{i}] Title: {t}")

    ctx_para = context_map.get(t)
    if ctx_para:
        ctx_snip = repr(ctx_para)
        if len(ctx_snip) > max_chars:
            ctx_snip = ctx_snip[:max_chars] + "..."
        print("    Context snippet:")
        print("    " + ctx_snip)
    else:
        print("    Context snippet: <no context paragraph for this title>")

    doc_snip = repr(d if isinstance(d, str) else str(d))
    if len(doc_snip) > max_chars:
        doc_snip = doc_snip[:max_chars] + "..."
    print("    Doc snippet:")
    print("    " + doc_snip)

    print("-" * 80)


Random example index: 320
_id:      483045fe0baf11ebab90acde48001122
type:     inference
level:    None
question: Who is Titus Tarquinius's paternal grandfather?
answer:   Lucius Tarquinius Priscus
Supporting facts:
[['Titus Tarquinius', 0], ['Lucius Tarquinius Superbus', 3]]
Titles, context paragraphs and docs (aligned by title):

[0] Title: Titus Tarquinius
    Context snippet:
    "Titus was the eldest son of Lucius Tarquinius Superbus, the last king of Rome. During his father's reign, he accompanied his younger brother Aruns and his cousin Lucius Junius Brutus to consult the Oracle at Delphi to have interpreted an omen witnessed by the king. In 509 BC, upon the overthrow of the monarchy, Titus went into exile at Caere with his father and his brother Aruns. In around 496 BC he fought with his father and the Latins against Rome at the Battle of Lake Regillus. During the battle, Marcus Valerius Volusus, who had been the Roman consul in 505 BC, charged Titus in an attempt to slay him, 

In [4]:
# 4: Helper functions for docs_chunks, docs, title_chunks, and supports

from typing import List, Dict, Any
import re

def align_to_titles(titles: List[str], seq_any: Any, fill_value: Any) -> List[Any]:
    # Pad/truncate seq to match titles length
    seq = list(seq_any) if isinstance(seq_any, list) else []
    if len(seq) < len(titles):
        seq = seq + [fill_value] * (len(titles) - len(seq))
    elif len(seq) > len(titles):
        seq = seq[:len(titles)]
    return seq

def build_context_map(ex: Dict[str, Any]) -> Dict[str, List[str]]:
    # Map each title to its context sentences from "context"
    ctx: Dict[str, List[str]] = {}
    for item in ex.get("context", []):
        if isinstance(item, (list, tuple)) and len(item) == 2:
            title, sent_list = item
            if isinstance(title, str) and isinstance(sent_list, list):
                sentences = [s.strip() for s in sent_list if isinstance(s, str) and s.strip()]
                ctx.setdefault(title, []).extend(sentences)
    return ctx

def build_docs2_map(ex: Dict[str, Any]) -> Dict[str, List[str]]:
    # Map each title to its docs2 sentence list (aligned by titles)
    titles = ex.get("titles", [])
    if not isinstance(titles, list): titles = []
    docs2 = align_to_titles(titles, ex.get("docs2", []), [])
    out: Dict[str, List[str]] = {}
    for t, sent_list in zip(titles, docs2):
        if isinstance(t, str) and isinstance(sent_list, list):
            out[t] = [s.strip() for s in sent_list if isinstance(s, str) and s.strip()]
        elif isinstance(t, str):
            out[t] = []
    return out

def make_docs_chunks(ex: Dict[str, Any]) -> List[List[str]]:
    # For each title, collect its context sentences (or [] if none)
    titles = ex.get("titles", [])
    if not isinstance(titles, list): titles = []
    ctx_map = build_context_map(ex)
    return [list(ctx_map.get(t, [])) for t in titles]

def make_docs_with_paragraph_newlines(ex: Dict[str, Any]) -> List[str]:
    """
    Build docs so that:
      - Each doc remains a single string in the list.
      - Paragraph boundaries kept but blank lines collapsed.
      - No '\n\n' in final docs (single '\n' only).
    """
    titles = ex.get("titles", [])
    if not isinstance(titles, list): titles = []
    docs_in = align_to_titles(titles, ex.get("docs", []), "")

    out_docs: List[str] = []
    for d in docs_in:
        if not isinstance(d, str):
            d = "" if d is None else str(d)

        text = d.replace("\r\n", "\n").replace("\r", "\n")
        text = re.sub(r"\n\s*\n+", "\n", text)  # collapse blank lines
        text = text.strip()
        out_docs.append(text)

    return out_docs

def make_title_chunks(ex: Dict[str, Any]) -> List[List[str]]:
    # Build [title, sentence] pairs from docs2 (aligned by titles)
    titles = ex.get("titles", [])
    if not isinstance(titles, list): titles = []
    docs2 = align_to_titles(titles, ex.get("docs2", []), [])

    out: List[List[str]] = []
    for t, sent_list in zip(titles, docs2):
        if isinstance(t, str) and isinstance(sent_list, list):
            for s in sent_list:
                if isinstance(s, str) and s.strip():
                    out.append([t, s.strip()])
    return out

def make_supports(ex: Dict[str, Any]) -> List[List[str]]:
    # Build [title, sentence] pairs from supporting_facts + (context, fallback docs2)
    ctx_map = build_context_map(ex)
    d2_map = build_docs2_map(ex)
    out: List[List[str]] = []

    for item in ex.get("supporting_facts", []):
        if not (isinstance(item, (list, tuple)) and len(item) == 2):
            continue
        title, idx = item
        if not isinstance(title, str) or not isinstance(idx, int):
            continue

        sent_list = ctx_map.get(title, [])
        if 0 <= idx < len(sent_list):
            s = sent_list[idx].strip()
            if s:
                out.append([title, s])
            continue

        # Fallback: if context missing/out-of-range, try docs2
        sent_list2 = d2_map.get(title, [])
        if 0 <= idx < len(sent_list2):
            s = sent_list2[idx].strip()
            if s:
                out.append([title, s])

    return out


In [5]:
# 5: Build transformed examples with the desired structure (+ add "type" after "answer")

from typing import List, Dict, Any

new_examples: List[Dict[str, Any]] = []

mismatch_docs = 0
mismatch_docs2 = 0

for ex in data:
    titles = ex.get("titles", [])
    if not isinstance(titles, list):
        titles = []

    docs_in = ex.get("docs", [])
    docs2_in = ex.get("docs2", [])

    if isinstance(docs_in, list) and len(docs_in) != len(titles):
        mismatch_docs += 1
    if isinstance(docs2_in, list) and len(docs2_in) != len(titles):
        mismatch_docs2 += 1

    # NOTE: dict insertion order is preserved in Python 3.7+,
    # so putting "type" right after "answer" will keep that order in the saved JSON.
    new_ex = {
        "question": ex.get("question"),
        "answer": ex.get("answer"),
        "type": ex.get("type"),
        "titles": list(titles),                       # keep titles order
        "docs_chunks": make_docs_chunks(ex),          # sentence-level from context
        "docs": make_docs_with_paragraph_newlines(ex),# docs normalized to single '\n'
        "title_chunks": make_title_chunks(ex),        # [title, sentence] from docs2
        "supports": make_supports(ex),                # [title, sentence] from supporting_facts
    }
    new_examples.append(new_ex)

print(f"Built {len(new_examples)} transformed examples.")
print(f"Examples with len(docs) != len(titles):  {mismatch_docs}")
print(f"Examples with len(docs2) != len(titles): {mismatch_docs2}")

# Quick sanity check on the first example
ex0 = new_examples[0]
print("Keys:", list(ex0.keys()))
print("len(titles):      ", len(ex0["titles"]))
print("len(docs_chunks): ", len(ex0["docs_chunks"]))
print("len(docs):        ", len(ex0["docs"]))
print("len(title_chunks):", len(ex0["title_chunks"]))
print("len(supports):    ", len(ex0["supports"]))
print("type:", ex0.get("type"))

Built 1000 transformed examples.
Examples with len(docs) != len(titles):  0
Examples with len(docs2) != len(titles): 0
Keys: ['question', 'answer', 'type', 'titles', 'docs_chunks', 'docs', 'title_chunks', 'supports']
len(titles):       10
len(docs_chunks):  10
len(docs):         10
len(title_chunks): 223
len(supports):     2
type: comparison


In [6]:
# 6: Save the transformed JSON into Google Drive under /content/drive/MyDrive/final_project

import json
from pathlib import Path

output_dir = Path("/content/drive/MyDrive/final_project")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "2wikimultihopqa_dev_2020wiki_1000_converted.json"

with output_path.open("w", encoding="utf-8") as f:
    json.dump(new_examples, f, ensure_ascii=False, indent=2)

print(f"Saved converted JSON to: {output_path}")


Saved converted JSON to: /content/drive/MyDrive/final_project/2wikimultihopqa_dev_2020wiki_1000_converted.json


In [7]:
# 7: Show first example from the original NEW JSON

import json

if not data:
    print("No examples in original JSON.")
else:
    print("=== Original NEW JSON: data[0] ===")
    print(json.dumps(data[0], ensure_ascii=False, indent=2))


=== Original NEW JSON: data[0] ===
{
  "_id": "a7b9672009c311ebbdb0ac1f6bf848b6",
  "question": "Are North Marion High School (Oregon) and Seoul High School both located in the same country?",
  "answer": "no",
  "type": "comparison",
  "level": null,
  "supporting_facts": [
    [
      "North Marion High School (Oregon)",
      0
    ],
    [
      "Seoul High School",
      0
    ]
  ],
  "context": [
    [
      "Calloway County High School",
      [
        "Calloway County",
        "High School is a public high school located in Murray, Kentucky.",
        "The school was formed from the consolidation of six high schools from across the county: Hazel High School, Lynn Grove High School, Kirksey High School, Almo High School, New Concord High School, and Faxon High School."
      ]
    ],
    [
      "Marion High School (Kansas)",
      [
        "Marion High School is a public high school in Marion, Kansas, USA.",
        "It is one of three schools operated by Marion USD 408, an

In [8]:
# 8: Show first example from the converted JSON we just wrote

import json
from pathlib import Path

conv_path = Path("/content/drive/MyDrive/final_project/2wikimultihopqa_dev_2020wiki_1000_converted.json")
assert conv_path.exists(), f"Converted file not found: {conv_path}"

with conv_path.open("r", encoding="utf-8") as f:
    converted = json.load(f)

print(f"Number of converted examples: {len(converted)}")

print("=== Converted JSON: converted[0] ===")
print(json.dumps(converted[0], ensure_ascii=False, indent=2))


Number of converted examples: 1000
=== Converted JSON: converted[0] ===
{
  "question": "Are North Marion High School (Oregon) and Seoul High School both located in the same country?",
  "answer": "no",
  "type": "comparison",
  "titles": [
    "Calloway County High School",
    "Marion High School (Kansas)",
    "North Marion High School (West Virginia)",
    "Creswell High School (Oregon)",
    "Wheeling High School",
    "Ottawa High School and Junior High School",
    "East High School (Denver)",
    "Seoul High School",
    "Marion High School (Indiana)",
    "North Marion High School (Oregon)"
  ],
  "docs_chunks": [
    [
      "Calloway County",
      "High School is a public high school located in Murray, Kentucky.",
      "The school was formed from the consolidation of six high schools from across the county: Hazel High School, Lynn Grove High School, Kirksey High School, Almo High School, New Concord High School, and Faxon High School."
    ],
    [
      "Marion High Schoo